# Unduh Indikator Ekonomi Makro

Mengunduh indikator ekonomi makro lewat API publik, **hanya harga emas Antam dan BI-Rate yang manual** (file sudah ada di `data/raw/`).

**Fallback:** kalau API sedang tidak bisa diakses, notebook otomatis memakai salinan cache di `data/raw/_fallback_cache/` (data historis dari eksperimen sebelumnya, sampai Agustus 2026) supaya notebook 01 dst. tetap bisa jalan.

In [1]:
import os
import time
import pandas as pd

RAW = os.path.join('..', 'data', 'raw')
FALLBACK = os.path.join(RAW, '_fallback_cache')
os.makedirs(RAW, exist_ok=True)

START = '2009-06-01'
END = '2026-08-31'


## Fungsi Unduh (dengan fallback ke cache)

In [2]:
def _get_yf(ticker):
    import yfinance as yf
    for attempt in range(3):
        try:
            data = yf.download(ticker, start=START, end=END, auto_adjust=False, progress=False)
            if data is not None and len(data) > 0:
                col = 'Adj Close' if 'Adj Close' in data.columns else 'Close'
                s = data[col]
                if isinstance(s, pd.DataFrame):
                    s = s.iloc[:, 0]
                return s.dropna()
        except Exception:
            time.sleep(2)
    return None


def _get_fred(series_id):
    import requests
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            parts = r.text.strip().splitlines()
            hdr = parts[0].split(',')
            date_col = hdr[0]
            rows = [tuple(line.split(',')) for line in parts[1:] if line.strip() and len(line.split(',')) >= 2]
            df = pd.DataFrame(rows, columns=[date_col, 'value'])
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            s = df.dropna().set_index(date_col)['value']
            return s[s.index >= START]
        except Exception as e:
            print(f'    (retry FRED {series_id}: {e})')
            time.sleep(2)
    return None


def simpan(nama, s):
    if s is None or len(s) == 0:
        cache_path = os.path.join(FALLBACK, f'{nama}.csv')
        if not os.path.exists(cache_path):
            print(f'[GAGAL] {nama}: unduhan kosong dan tidak ada cache fallback '
                  f'(indikator ini tidak dipakai di data_gabungan.csv, aman diabaikan '
                  f'kalau memang bukan salah satu dari 7 indikator utama)')
            return
        cached = pd.read_csv(cache_path, index_col=0, parse_dates=True).iloc[:, 0]
        cached.rename(nama).to_frame().to_csv(os.path.join(RAW, f'{nama}.csv'))
        print(f'[FALLBACK] {nama}: unduhan gagal/kosong, pakai cache '
              f'({len(cached)} baris, s.d. {cached.index.max().date()})')
        return
    s = s[~s.index.duplicated(keep='last')].sort_index()
    s.rename(nama).to_frame().to_csv(os.path.join(RAW, f'{nama}.csv'))
    print(f'[OK] {nama}: {len(s)} baris | {s.index.min().date()} s.d. {s.index.max().date()}')


## 1. Yahoo Finance: DXY, S&P 500, USD/IDR, IHSG

In [3]:
TICKERS = [('dxy', 'DX-Y.NYB'), ('sp500', '^GSPC'), ('usd_idr', 'USDIDR=X'), ('ihsg', '^JKSE')]
for nama, tkr in TICKERS:
    simpan(nama, _get_yf(tkr))
    time.sleep(0.5)


[OK] dxy: 4340 baris | 2009-06-01 s.d. 2026-08-28
[OK] sp500: 4339 baris | 2009-06-01 s.d. 2026-08-28
[OK] usd_idr: 4489 baris | 2009-06-01 s.d. 2026-08-28
[OK] ihsg: 4180 baris | 2009-06-01 s.d. 2026-08-28


## 2. FRED: US 10-Year Treasury Yield, Federal Funds Rate, dan id_tbill (tidak dipakai, hanya kelengkapan)

In [4]:
FRED_SERIES = [('us10yr', 'DGS10'), ('ffr', 'FEDFUNDS'), ('id_tbill', 'INTDSRIDM193N')]
for nama, sid in FRED_SERIES:
    simpan(nama, _get_fred(sid))
    time.sleep(0.5)


[OK] us10yr: 4346 baris | 2009-06-01 s.d. 2026-10-14
[OK] ffr: 208 baris | 2009-06-01 s.d. 2026-09-01
[OK] id_tbill: 47 baris | 2009-06-01 s.d. 2013-04-01


## Ringkasan

In [5]:
print('Isi data/raw/ untuk indikator yang berhasil diunduh/di-fallback:')
for nama, _ in TICKERS + FRED_SERIES:
    path = os.path.join(RAW, f'{nama}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        print(f'  {nama:10s}: {len(df):5d} baris | {df.index.min().date()} s.d. {df.index.max().date()}')
    else:
        print(f'  {nama:10s}: TIDAK ADA — cek error di atas')


Isi data/raw/ untuk indikator yang berhasil diunduh/di-fallback:
  dxy       :  4340 baris | 2009-06-01 s.d. 2026-08-28
  sp500     :  4339 baris | 2009-06-01 s.d. 2026-08-28
  usd_idr   :  4489 baris | 2009-06-01 s.d. 2026-08-28
  ihsg      :  4180 baris | 2009-06-01 s.d. 2026-08-28
  us10yr    :  4346 baris | 2009-06-01 s.d. 2026-10-14
  ffr       :   208 baris | 2009-06-01 s.d. 2026-09-01
  id_tbill  :    47 baris | 2009-06-01 s.d. 2013-04-01
